# Preparing Environment

In [1]:
import numpy as np
import pandas as pd
import openpyxl
import re
import umap
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 200)

In [2]:
covariates = ['sex', 'HTN', 'DM', 'CKD', 'CLD', 'Asthma', 'COPD', 'CLD (chronic lung disease)', 'Stroke', 
              'Spinal Cord Injury', 'Neuro (Parkinsons, Epilepsy, Dementia)', 'Obesity', 'Immunodeficiency ', 
              'Cancer', 'Anxiety', 'Depression', 'Psych', 'Eating disorder', 'IBD/IBS']
symptoms = ['Abnormal Body Temperature', 'Unexplained sweat/flushing', 'Difficult / Labored Breathing, SoB', 
            'Breathing faster than normal ', 'Fatigue (1)', 'Weakness', 'Postexertional Malaise (7)', 'Cough', 
            'Chronic Cough (4)', 'Dizziness (1)', 'Fainting/blackouts', 'Headache', 'Joint Pain arthralgia', 
            'Chest Pain (2)', 'Paliptations (2)', 'Brain Fog (3)', 'Confusion, difficulty thinking', 
            'Disoriented, getting lost', 'Forgetful, memory problem', 'Loss of Taste', 'Loss of Smell', 
            'Difficulty swallowing', 'Loss of Appetite/weight loss', 'Dry Mouth', 'Thirst (3)', 'Dry Eye', 
            'Rhinitis', 'Sore throat', 'Diarrhea', 'Nausea', 'Vomitting', 'Acid Reflux/indigestion', 'Bloating', 
            'Abdominal Pain', 'Muscle Pain', 'Tingling/numbness in the mouth/face', 
            'Tingling/Numbness burning, stabbing, pin and needle', 'Sleep Disturbace', 'Swelling', 
            'Urinary incontinence ', 'Menstrual irregularity ', 'Hair Loss (0)', 'Rashes/ skin lesions', 
            'tinnitus', 'Difficulty hearing', 'Vision effects/double blurry vision ', 'Fever/Chills', 'Anxiety', 
            'Mood swing/irritabiity, Depression', 'Exaggerated sx or worse from alcohol ', 
            'Sexual desire or capacity (1)', 'Abnormal Movements (1)']
other = ['RBD MENSA IgG', 'S1 MENSA IgG', 'S2 MENSA IgG', 'NP MENSA IgG', 'CMV_gB_1 MENSA IgG', 'CMV_Pentamer MENSA IgG', 
         'EBV_EBNA1 MENSA IgG', 'EBV_VCA MENSA IgG', 'EBV_gp350 MENSA IgG', 'HSV2_gD MENSA IgG', 'RBD Serum IgG', 
         'S1 Serum IgG', 'S2 Serum IgG', 'NP Serum IgG', 'CMV_gB_1 Serum IgG', 'CMV_Pentamer Serum IgG', 
         'EBV_EBNA1 Serum IgG', 'EBV_VCA Serum IgG', 'EBV_gp350 Serum IgG', 'HSV2_gD Serum IgG']

# Symptom mapping to sinai

In [3]:
data_dictionary = pd.read_csv("data_dictionary - Sheet1.csv").astype(str)
data_dictionary = data_dictionary.replace("currentsymptoms_7\r", "currentsymptoms_7")
data_dictionary = data_dictionary[data_dictionary["Sinai Name"] != "nan"]
data_dictionary = data_dictionary[data_dictionary["Emory Name"] != "nan"]
data_dictionary = dict(zip(data_dictionary["Emory Name"], data_dictionary["Sinai Name"]))
data_dictionary

{'Brain Fog (3)': 'currentsymptoms_27',
 'Confusion, difficulty thinking': 'currentsymptoms_28',
 'Forgetful, memory problem': 'currentsymptoms_30',
 'Fatigue (1)': 'currentsymptoms_15',
 'Cough': 'currentsymptoms_11',
 'Difficult / Labored Breathing, SoB': 'currentsymptoms_13',
 'Breathing faster than normal': 'currentsymptoms_14',
 'Headache': 'currentsymptoms_2',
 'Sleep Disturbace': 'currentsymptoms_42',
 'Weakness': 'currentsymptoms_31',
 'Joint Pain arthralgia': 'currentsymptoms_33',
 'Loss of Appetite/weight loss': 'currentsymptoms_24',
 'Nausea': 'currentsymptoms_21',
 'Chest Pain (2)': 'currentsymptoms_16',
 'Paliptations (2)': 'currentsymptoms_17',
 'Loss of Smell': 'currentsymptoms_8',
 'Muscle Pain': 'currentsymptoms_32',
 'Rashes/ skin lesions': 'currentsymptoms_41',
 'Dizziness (1)': 'currentsymptoms_18',
 'Tingling/Numbness burning, stabbing, pin and needle': 'currentsymptoms_35',
 'Vision effects/double blurry vision': 'currentsymptoms_5',
 'Sore throat': 'currentsympto

In [4]:
symptoms = ['Abnormal Body Temperature', 'Unexplained sweat/flushing', 'Difficult / Labored Breathing, SoB', 'Breathing faster than normal ', 'Fatigue (1)', 'Weakness', 'Postexertional Malaise (7)', 'Cough', 'Chronic Cough (4)', 'Dizziness (1)', 'Fainting/blackouts', 'Headache', 'Joint Pain arthralgia', 'Chest Pain (2)', 'Paliptations (2)', 'Brain Fog (3)', 'Confusion, difficulty thinking', 'Disoriented, getting lost', 'Forgetful, memory problem', 'Loss of Taste', 'Loss of Smell', 'Difficulty swallowing', 'Loss of Appetite/weight loss', 'Dry Mouth', 'Thirst (3)', 'Dry Eye', 'Rhinitis', 'Sore throat', 'Diarrhea', 'Nausea', 'Vomitting', 'Acid Reflux/indigestion', 'Bloating', 'Abdominal Pain', 'Muscle Pain', 'Tingling/numbness in the mouth/face', 'Tingling/Numbness burning, stabbing, pin and needle', 'Sleep Disturbace', 'Swelling', 'Urinary incontinence ', 'Menstrual irregularity ', 'Hair Loss (0)', 'Rashes/ skin lesions', 'tinnitus', 'Difficulty hearing', 'Vision effects/double blurry vision ', 'Fever/Chills', 'Anxiety', 'Mood swing/irritabiity, Depression', 'Exaggerated sx or worse from alcohol ', 'Sexual desire or capacity (1)', 'Abnormal Movements (1)']
symptoms_new = []
for s in symptoms: 
    s = s.lower().replace(" ", "_").replace("/", "_").replace(",", "_").replace("(", "_").replace(")", "_")
    while "__" in s: 
        s = s.replace("__", "_")
    if s[-1] == "_": s = s[:-1]
    symptoms_new.append(s)
symptoms_dict = dict(zip(symptoms, symptoms_new))
symptoms_dict

{'Abnormal Body Temperature': 'abnormal_body_temperature',
 'Unexplained sweat/flushing': 'unexplained_sweat_flushing',
 'Difficult / Labored Breathing, SoB': 'difficult_labored_breathing_sob',
 'Breathing faster than normal ': 'breathing_faster_than_normal',
 'Fatigue (1)': 'fatigue_1',
 'Weakness': 'weakness',
 'Postexertional Malaise (7)': 'postexertional_malaise_7',
 'Cough': 'cough',
 'Chronic Cough (4)': 'chronic_cough_4',
 'Dizziness (1)': 'dizziness_1',
 'Fainting/blackouts': 'fainting_blackouts',
 'Headache': 'headache',
 'Joint Pain arthralgia': 'joint_pain_arthralgia',
 'Chest Pain (2)': 'chest_pain_2',
 'Paliptations (2)': 'paliptations_2',
 'Brain Fog (3)': 'brain_fog_3',
 'Confusion, difficulty thinking': 'confusion_difficulty_thinking',
 'Disoriented, getting lost': 'disoriented_getting_lost',
 'Forgetful, memory problem': 'forgetful_memory_problem',
 'Loss of Taste': 'loss_of_taste',
 'Loss of Smell': 'loss_of_smell',
 'Difficulty swallowing': 'difficulty_swallowing',
 

# Loading Data

In [5]:
file = "data_from_collaborators/Data Collection for Sinai_v2.xlsx"
file = openpyxl.load_workbook(file, data_only = True)
print(file.sheetnames)

['Sheet1']


In [6]:
# Getting raw data
sheet = "Sheet1"
data = pd.DataFrame(file[sheet].values).astype(str)
data = data.fillna(float("nan"))
data.columns = [str(val) for val in data.iloc[0,:]]
data = data.iloc[1:, :]
data = data[data["Subject #"] != "3477"].reset_index(drop = True)
keep = [0, 2]
keep.extend(range(10, 28))
keep.extend(range(38, 90))
first = 0
last = 0
igg_cols = []
for i in range(data.shape[1]): 
    col = list(data.columns)[i]
    if "serum" in col.lower() or "mensa" in col.lower(): 
#         print(col)
        data = data.rename(columns = {col: col.lower().replace(" ", "_")})
        col = col.lower().replace(" ", "_")
        igg_cols.append(col)
        if first == 0: 
            first = i
        last = i
keep.extend(range(first, last + 1))
data = data.iloc[:,keep]
data = data.rename(columns = {"Gender ": "sex", "Exaggerated sx or worse from alcohol ":"Exaggerated sx or worse from alcohol"})
data = data.rename(columns = data_dictionary)
data = data.replace("3579 3582", 3582)
data.index = data["Subject #"]
print(data.shape)
data

(60, 92)


,Subject #,sex,HTN,DM,CKD,CLD,Asthma,COPD,CLD (chronic lung disease),Stroke,Spinal Cord Injury,"Neuro (Parkinsons, Epilepsy, Dementia)",Obesity,Immunodeficiency,Cancer,Anxiety,Depression,Psych,Eating disorder,IBD/IBS,currentsymptoms_3,currentsymptoms_4,currentsymptoms_13,Breathing faster than normal,currentsymptoms_15,currentsymptoms_31,Postexertional Malaise (7),currentsymptoms_11,Chronic Cough (4),currentsymptoms_18,currentsymptoms_19,currentsymptoms_2,currentsymptoms_33,currentsymptoms_16,currentsymptoms_17,currentsymptoms_27,currentsymptoms_28,currentsymptoms_29,currentsymptoms_30,currentsymptoms_9,currentsymptoms_8,currentsymptoms_10,currentsymptoms_24,Dry Mouth,Thirst (3),Dry Eye,Rhinitis,currentsymptoms_12,currentsymptoms_20,currentsymptoms_21,currentsymptoms_22,currentsymptoms_23,currentsymptoms_25,currentsymptoms_26,currentsymptoms_32,currentsymptoms_34,currentsymptoms_35,currentsymptoms_42,currentsymptoms_36,Urinary incontinence,Menstrual irregularity,currentsymptoms_40,currentsymptoms_41,currentsymptoms_7,currentsymptoms_6,Vision effects/double blurry vision,currentsymptoms_1,Anxiety_2,currentsymptoms_43,currentsymptoms_44,currentsymptoms_39,Abnormal Movements (1),rbd_mensa_igg,s1_mensa_igg,s2_mensa_igg,np_mensa_igg,cmv_gb_1_mensa_igg,cmv_pentamer_mensa_igg,ebv_ebna1_mensa_igg,ebv_vca_mensa_igg,ebv_gp350_mensa_igg,hsv2_gd_mensa_igg,rbd_serum_igg,s1_serum_igg,s2_serum_igg,np_serum_igg,cmv_gb_1_serum_igg,cmv_pentamer_serum_igg,ebv_ebna1_serum_igg,ebv_vca_serum_igg,ebv_gp350_serum_igg,hsv2_gd_serum_igg
Subject #,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3633,3633,F,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1,1,1,0,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,1,1,0,0,0,1,1,0,0,1,61.25,84.5,59,11.75,25.5,1,7.25,39.25,0.75,-8.5,197923,181317,187498.5,27079.25,295,252.75,74592.5,2029,213.5,57061.75
7191,7191,F,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0,1,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,1,0,0,0,186,118.5,193.25,27,127.5,48.75,112.5,81.5,37,92.5,137940.75,112925,154675,43216.5,80568.75,32537.5,6770.5,46710,19973.75,90179.75
7283,7283,M,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,1,0,0,1,1,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,593.5,426.25,376.25,-9.75,163,132.5,66.75,21.5,14.5,-22.75,215546,211708,227075.5,547.25,114837.75,54140,92600.75,12822.75,352.75,399.25
7385,7385,M,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,1,0,0,0,1,0,1,1,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,0,1,1,0,0,0,0,0,223.25,190.25,235,17,197.75,87,130,493.5,156.5,51.25,182501,166949.25,188780,3507.5,113108.5,61596.25,127289.5,186401.5,114430.25,83217.5
7415,7415,F,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,1,1,0,1,1,0,1,1,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,1,0,0,0,1113,685.25,665.75,22.75,17.5,7,20,90.75,105.5,11,180698,199563,229105.25,1575,476.5,367.5,25938,32915,45131.25,209.75
7423,7423,F,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,1,0,0,1,1,1,0,1,1,1,0,1,0,0,0,1,1,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,1,0,0,0,258.5,166.5,282.5,-1,36,22.25,10.25,270,52.5,207,174829.75,144314.25,205137.75,562,257.25,134.5,68627.5,108870,64012.25,128146.5
7424,7424,F,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,1,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,1,1,0,0,0,1893.5,1323.25,2509.75,-5,115.5,100.25,-11.25,637,1649.5,283.5,174166,201581.5,206533,15292.75,66127,48691.5,6081,157824.75,197410.5,127998.75
7426,7426,M,0,0,0,0,1,0,1,0,0,1,1,0,0,0,0,0,0,0,0,1,1,0,1,1,0,0,0,1,1,1,1,0,0,1,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,956.75,640.25,393.75,45.25,90.25,124.25,-10,64.25,-21.5,43.75,189378.75,191103,187612,33458.5,54611.25,73298.75,50223,18252,7954.25,58212.75
7010,7010,F,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,1,1,1,1,1,0,1,1,1,1,1,0,0,1,0,0

In [7]:
columns = list(filter(re.compile("currentsymptoms_.*").match, data.columns))
columns = [int(val.split("_")[-1]) for val in columns]
columns = sorted(columns)
columns = [f"currentsymptoms_{i}" for i in columns]
columns_ = covariates[:]
columns_.extend(columns)
columns = columns_[:]
columns.extend(igg_cols)
data = data[columns]
data

,sex,HTN,DM,CKD,CLD,Asthma,COPD,CLD (chronic lung disease),Stroke,Spinal Cord Injury,"Neuro (Parkinsons, Epilepsy, Dementia)",Obesity,Immunodeficiency,Cancer,Anxiety,Depression,Psych,Eating disorder,IBD/IBS,currentsymptoms_1,currentsymptoms_2,currentsymptoms_3,currentsymptoms_4,currentsymptoms_6,currentsymptoms_7,currentsymptoms_8,currentsymptoms_9,currentsymptoms_10,currentsymptoms_11,currentsymptoms_12,currentsymptoms_13,currentsymptoms_15,currentsymptoms_16,currentsymptoms_17,currentsymptoms_18,currentsymptoms_19,currentsymptoms_20,currentsymptoms_21,currentsymptoms_22,currentsymptoms_23,currentsymptoms_24,currentsymptoms_25,currentsymptoms_26,currentsymptoms_27,currentsymptoms_28,currentsymptoms_29,currentsymptoms_30,currentsymptoms_31,currentsymptoms_32,currentsymptoms_33,currentsymptoms_34,currentsymptoms_35,currentsymptoms_36,currentsymptoms_39,currentsymptoms_40,currentsymptoms_41,currentsymptoms_42,currentsymptoms_43,currentsymptoms_44,rbd_mensa_igg,s1_mensa_igg,s2_mensa_igg,np_mensa_igg,cmv_gb_1_mensa_igg,cmv_pentamer_mensa_igg,ebv_ebna1_mensa_igg,ebv_vca_mensa_igg,ebv_gp350_mensa_igg,hsv2_gd_mensa_igg,rbd_serum_igg,s1_serum_igg,s2_serum_igg,np_serum_igg,cmv_gb_1_serum_igg,cmv_pentamer_serum_igg,ebv_ebna1_serum_igg,ebv_vca_serum_igg,ebv_gp350_serum_igg,hsv2_gd_serum_igg
Subject #,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3633,F,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,1,1,1,1,0,0,0,0,0,0,0,0,1,1,0,0,0,1,1,0,0,0,0,1,1,1,1,0,61.25,84.5,59,11.75,25.5,1,7.25,39.25,0.75,-8.5,197923,181317,187498.5,27079.25,295,252.75,74592.5,2029,213.5,57061.75
7191,F,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,1,0,186,118.5,193.25,27,127.5,48.75,112.5,81.5,37,92.5,137940.75,112925,154675,43216.5,80568.75,32537.5,6770.5,46710,19973.75,90179.75
7283,M,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,0,0,0,0,1,0,0,0,1,1,0,1,1,1,1,0,0,0,0,1,0,1,1,0,593.5,426.25,376.25,-9.75,163,132.5,66.75,21.5,14.5,-22.75,215546,211708,227075.5,547.25,114837.75,54140,92600.75,12822.75,352.75,399.25
7385,M,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,1,0,0,1,1,0,0,1,1,1,0,0,1,0,1,0,1,0,0,223.25,190.25,235,17,197.75,87,130,493.5,156.5,51.25,182501,166949.25,188780,3507.5,113108.5,61596.25,127289.5,186401.5,114430.25,83217.5
7415,F,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,1,1,0,1,1,0,0,1,0,0,1,0,0,1,1,0,0,0,0,1,0,0,0,0,1,0,1,1,0,1113,685.25,665.75,22.75,17.5,7,20,90.75,105.5,11,180698,199563,229105.25,1575,476.5,367.5,25938,32915,45131.25,209.75
7423,F,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,0,1,1,1,1,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,1,1,0,258.5,166.5,282.5,-1,36,22.25,10.25,270,52.5,207,174829.75,144314.25,205137.75,562,257.25,134.5,68627.5,108870,64012.25,128146.5
7424,F,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,1,0,1,1,0,1893.5,1323.25,2509.75,-5,115.5,100.25,-11.25,637,1649.5,283.5,174166,201581.5,206533,15292.75,66127,48691.5,6081,157824.75,197410.5,127998.75
7426,M,0,0,0,0,1,0,1,0,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,1,1,0,0,1,1,0,0,0,0,1,0,0,1,1,0,0,1,1,1,0,0,1,0,0,0,1,0,0,956.75,640.25,393.75,45.25,90.25,124.25,-10,64.25,-21.5,43.75,189378.75,191103,187612,33458.5,54611.25,73298.75,50223,18252,7954.25,58212.75
7010,F,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,1,1,0,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,1,1,0,1,0,0,0,0,1,1,0,201.75,112.5,176.25,-18.25,-32.75,37.5,1126.25,2836.25,65.5,58.25,53927.75,33040.25,51370.75,1669,77.75,202.5,223942.75,182973.5,19613.5,16821


In [8]:
# data.to_csv("data_cleaned/data_collection_for_sinai.csv", index = True)

In [9]:
comorbs = ['HTN', 'DM', 'CKD', 'CLD', 'Asthma', 'COPD', 'CLD (chronic lung disease)', 'Stroke', 'Spinal Cord Injury', 'Neuro (Parkinsons, Epilepsy, Dementia)', 'Obesity', 'Immunodeficiency ', 'Cancer', 'Anxiety', 'Depression', 'Psych', 'Eating disorder', 'IBD/IBS']
comorb = ["Hypertension", "Diabetes", "Chronic Kidney Disease", "Chronic Liver Disease (i.e Cirrhosis, Hepatitis B)", "Asthma", "Chronic Obstructive Pulmonary Disease", "Other Chronic Lung Disease", "Stroke", "Spinal Cord Injury", "Other Neurological Disease (Parkinson, Epilepsy, Dementia)", "Obesity", "Immunodeficiency (i.e auto-immune disease)", "Cancer", "Anxiety", "Depression", "Other Psychiatric Disease", "Eating disorder", "Inflammatory Bowel Disease/ Irritable Bowel Syndrome", "Other - please specify below", "None"]
comorb_dict = dict(zip(comorbs, comorb))
data = data.rename(columns = comorb_dict)
comorb_dict = dict(zip(comorb, ["comorb_selfreport_" + str(i) for i in range(1, len(comorb) + 1)]))
data = data.rename(columns = comorb_dict)
# comorb_dict = dict(zip(["comorb_selfreport_" + str(i) for i in range(1, len(comorb) + 1)], comorb))
data = data.reset_index().rename(columns = {"Subject #": "record_id"})
data

,record_id,sex,comorb_selfreport_1,comorb_selfreport_2,comorb_selfreport_3,comorb_selfreport_4,comorb_selfreport_5,comorb_selfreport_6,comorb_selfreport_7,comorb_selfreport_8,comorb_selfreport_9,comorb_selfreport_10,comorb_selfreport_11,comorb_selfreport_12,comorb_selfreport_13,comorb_selfreport_14,comorb_selfreport_15,comorb_selfreport_16,comorb_selfreport_17,comorb_selfreport_18,currentsymptoms_1,currentsymptoms_2,currentsymptoms_3,currentsymptoms_4,currentsymptoms_6,currentsymptoms_7,currentsymptoms_8,currentsymptoms_9,currentsymptoms_10,currentsymptoms_11,currentsymptoms_12,currentsymptoms_13,currentsymptoms_15,currentsymptoms_16,currentsymptoms_17,currentsymptoms_18,currentsymptoms_19,currentsymptoms_20,currentsymptoms_21,currentsymptoms_22,currentsymptoms_23,currentsymptoms_24,currentsymptoms_25,currentsymptoms_26,currentsymptoms_27,currentsymptoms_28,currentsymptoms_29,currentsymptoms_30,currentsymptoms_31,currentsymptoms_32,currentsymptoms_33,currentsymptoms_34,currentsymptoms_35,currentsymptoms_36,currentsymptoms_39,currentsymptoms_40,currentsymptoms_41,currentsymptoms_42,currentsymptoms_43,currentsymptoms_44,rbd_mensa_igg,s1_mensa_igg,s2_mensa_igg,np_mensa_igg,cmv_gb_1_mensa_igg,cmv_pentamer_mensa_igg,ebv_ebna1_mensa_igg,ebv_vca_mensa_igg,ebv_gp350_mensa_igg,hsv2_gd_mensa_igg,rbd_serum_igg,s1_serum_igg,s2_serum_igg,np_serum_igg,cmv_gb_1_serum_igg,cmv_pentamer_serum_igg,ebv_ebna1_serum_igg,ebv_vca_serum_igg,ebv_gp350_serum_igg,hsv2_gd_serum_igg
0,3633,F,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,1,1,1,1,0,0,0,0,0,0,0,0,1,1,0,0,0,1,1,0,0,0,0,1,1,1,1,0,61.25,84.5,59,11.75,25.5,1,7.25,39.25,0.75,-8.5,197923,181317,187498.5,27079.25,295,252.75,74592.5,2029,213.5,57061.75
1,7191,F,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,1,0,186,118.5,193.25,27,127.5,48.75,112.5,81.5,37,92.5,137940.75,112925,154675,43216.5,80568.75,32537.5,6770.5,46710,19973.75,90179.75
2,7283,M,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,0,0,0,0,1,0,0,0,1,1,0,1,1,1,1,0,0,0,0,1,0,1,1,0,593.5,426.25,376.25,-9.75,163,132.5,66.75,21.5,14.5,-22.75,215546,211708,227075.5,547.25,114837.75,54140,92600.75,12822.75,352.75,399.25
3,7385,M,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,1,0,0,1,1,0,0,1,1,1,0,0,1,0,1,0,1,0,0,223.25,190.25,235,17,197.75,87,130,493.5,156.5,51.25,182501,166949.25,188780,3507.5,113108.5,61596.25,127289.5,186401.5,114430.25,83217.5
4,7415,F,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,1,1,0,1,1,0,0,1,0,0,1,0,0,1,1,0,0,0,0,1,0,0,0,0,1,0,1,1,0,1113,685.25,665.75,22.75,17.5,7,20,90.75,105.5,11,180698,199563,229105.25,1575,476.5,367.5,25938,32915,45131.25,209.75
5,7423,F,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,0,1,1,1,1,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,1,1,0,258.5,166.5,282.5,-1,36,22.25,10.25,270,52.5,207,174829.75,144314.25,205137.75,562,257.25,134.5,68627.5,108870,64012.25,128146.5
6,7424,F,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,1,0,1,1,0,1893.5,1323.25,2509.75,-5,115.5,100.25,-11.25,637,1649.5,283.5,174166,201581.5,206533,15292.75,66127,48691.5,6081,157824.75,197410.5,127998.75
7,7426,M,0,0,0,0,1,0,1,0,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,1,1,0,0,1,1,0,0,0,0,1,0,0,1,1,0,0,1,1,1,0,0,1,0,0,0,1,0,0,956.75,640.25,393.75,45.25,90.25,124.25,-10,64.25,-21.5,43.75,189378.75,191103,187612,33458.5,54611.25,73298.75,50223,18252,7954.25,58212.75
8,7010,F,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,1,1,0,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,1,1,0,1,0,0,0,0,1,1,0,201.75,112.5,176.25,-18.25,-32.75,37.5,1126.25,2836.25,65.5,58.25,53927.75,33040.25,51370.75,1669,77.75,202.5,223942.75,182973.5,19613.5,16821
9,7023,M,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1047,768,833.5,3.25,81.25,85.5,6,192.75,234,34.25,193681.25,209817.5,247

# Merge Sinai and Emory

In [10]:
# sinai = pd.read_csv("data_cleaned/UCSF_8_21_2024_DATAEXPORT_original_survey.csv").rename(columns = {"comorb_selfreport_v2___13": "comorb_selfreport_13"})
sinai = pd.read_csv("data_cleaned/UCSF_2_4_25_most_recent_surveys_100_n825.csv").rename(columns = {"comorb_selfreport_v2___13": "comorb_selfreport_13"})
sinai

,record_id,yale_id,sex,currentsymptoms_1,currentsymptoms_2,currentsymptoms_3,currentsymptoms_4,currentsymptoms_5,currentsymptoms_6,currentsymptoms_7,currentsymptoms_8,currentsymptoms_9,currentsymptoms_10,currentsymptoms_11,currentsymptoms_12,currentsymptoms_13,currentsymptoms_14,currentsymptoms_15,currentsymptoms_16,currentsymptoms_17,currentsymptoms_18,currentsymptoms_19,currentsymptoms_20,currentsymptoms_21,currentsymptoms_22,currentsymptoms_23,currentsymptoms_24,currentsymptoms_25,currentsymptoms_26,currentsymptoms_27,currentsymptoms_28,currentsymptoms_29,currentsymptoms_30,currentsymptoms_31,currentsymptoms_32,currentsymptoms_33,currentsymptoms_34,currentsymptoms_35,currentsymptoms_36,currentsymptoms_37,currentsymptoms_38,currentsymptoms_39,currentsymptoms_40,currentsymptoms_41,currentsymptoms_42,currentsymptoms_43,currentsymptoms_44,currentsymptoms_45,currentsymptoms_46,comorb_selfreport_1,comorb_selfreport_2,comorb_selfreport_3,comorb_selfreport_4,comorb_selfreport_5,comorb_selfreport_6,comorb_selfreport_7,comorb_selfreport_8,comorb_selfreport_9,comorb_selfreport_10,comorb_selfreport_11,comorb_selfreport_12,comorb_selfreport_13,comorb_selfreport_14,comorb_selfreport_15,comorb_selfreport_16,comorb_selfreport_17,comorb_selfreport_18,comorb_selfreport_19,comorb_selfreport_20,eq5vas
0,2,79,M,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,75.0
1,3,NaN,F,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,1,0,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0
2,7,NaN,M,0,1,0,0,0,1,1,0,0,0,0,0,1,1,1,0,0,1,0,1,1,0,1,0,0,1,1,1,1,1,1,1,0,1,1,0,1,0,1,0,0,1,1,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0
3,8,NaN,F,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0
4,9,NaN,F,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,1,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,1,0,1,0,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
820,1272,NaN,F,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,85.0
821,1273,NaN,M,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,0,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,7.0
822,1274,NaN,M,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,1,1,0,1,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,25.0
823,1275,NaN,F,0,1,0,1,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,1,1,1,0,1,0,0,1,1,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,71.0


In [11]:
# emory = pd.read_csv("data_cleaned/data_collection_for_sinai.csv").rename(columns = {"Subject #": "record_id"})
emory = data.copy()
emory

,record_id,sex,comorb_selfreport_1,comorb_selfreport_2,comorb_selfreport_3,comorb_selfreport_4,comorb_selfreport_5,comorb_selfreport_6,comorb_selfreport_7,comorb_selfreport_8,comorb_selfreport_9,comorb_selfreport_10,comorb_selfreport_11,comorb_selfreport_12,comorb_selfreport_13,comorb_selfreport_14,comorb_selfreport_15,comorb_selfreport_16,comorb_selfreport_17,comorb_selfreport_18,currentsymptoms_1,currentsymptoms_2,currentsymptoms_3,currentsymptoms_4,currentsymptoms_6,currentsymptoms_7,currentsymptoms_8,currentsymptoms_9,currentsymptoms_10,currentsymptoms_11,currentsymptoms_12,currentsymptoms_13,currentsymptoms_15,currentsymptoms_16,currentsymptoms_17,currentsymptoms_18,currentsymptoms_19,currentsymptoms_20,currentsymptoms_21,currentsymptoms_22,currentsymptoms_23,currentsymptoms_24,currentsymptoms_25,currentsymptoms_26,currentsymptoms_27,currentsymptoms_28,currentsymptoms_29,currentsymptoms_30,currentsymptoms_31,currentsymptoms_32,currentsymptoms_33,currentsymptoms_34,currentsymptoms_35,currentsymptoms_36,currentsymptoms_39,currentsymptoms_40,currentsymptoms_41,currentsymptoms_42,currentsymptoms_43,currentsymptoms_44,rbd_mensa_igg,s1_mensa_igg,s2_mensa_igg,np_mensa_igg,cmv_gb_1_mensa_igg,cmv_pentamer_mensa_igg,ebv_ebna1_mensa_igg,ebv_vca_mensa_igg,ebv_gp350_mensa_igg,hsv2_gd_mensa_igg,rbd_serum_igg,s1_serum_igg,s2_serum_igg,np_serum_igg,cmv_gb_1_serum_igg,cmv_pentamer_serum_igg,ebv_ebna1_serum_igg,ebv_vca_serum_igg,ebv_gp350_serum_igg,hsv2_gd_serum_igg
0,3633,F,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,1,1,1,1,0,0,0,0,0,0,0,0,1,1,0,0,0,1,1,0,0,0,0,1,1,1,1,0,61.25,84.5,59,11.75,25.5,1,7.25,39.25,0.75,-8.5,197923,181317,187498.5,27079.25,295,252.75,74592.5,2029,213.5,57061.75
1,7191,F,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,1,0,186,118.5,193.25,27,127.5,48.75,112.5,81.5,37,92.5,137940.75,112925,154675,43216.5,80568.75,32537.5,6770.5,46710,19973.75,90179.75
2,7283,M,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,1,1,1,0,0,0,0,1,0,0,0,1,1,0,1,1,1,1,0,0,0,0,1,0,1,1,0,593.5,426.25,376.25,-9.75,163,132.5,66.75,21.5,14.5,-22.75,215546,211708,227075.5,547.25,114837.75,54140,92600.75,12822.75,352.75,399.25
3,7385,M,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,1,0,0,1,1,0,0,1,1,1,0,0,1,0,1,0,1,0,0,223.25,190.25,235,17,197.75,87,130,493.5,156.5,51.25,182501,166949.25,188780,3507.5,113108.5,61596.25,127289.5,186401.5,114430.25,83217.5
4,7415,F,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,1,1,0,1,1,0,0,1,0,0,1,0,0,1,1,0,0,0,0,1,0,0,0,0,1,0,1,1,0,1113,685.25,665.75,22.75,17.5,7,20,90.75,105.5,11,180698,199563,229105.25,1575,476.5,367.5,25938,32915,45131.25,209.75
5,7423,F,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,0,1,1,1,1,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,1,1,0,258.5,166.5,282.5,-1,36,22.25,10.25,270,52.5,207,174829.75,144314.25,205137.75,562,257.25,134.5,68627.5,108870,64012.25,128146.5
6,7424,F,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,1,0,1,1,0,1893.5,1323.25,2509.75,-5,115.5,100.25,-11.25,637,1649.5,283.5,174166,201581.5,206533,15292.75,66127,48691.5,6081,157824.75,197410.5,127998.75
7,7426,M,0,0,0,0,1,0,1,0,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,1,1,0,0,1,1,0,0,0,0,1,0,0,1,1,0,0,1,1,1,0,0,1,0,0,0,1,0,0,956.75,640.25,393.75,45.25,90.25,124.25,-10,64.25,-21.5,43.75,189378.75,191103,187612,33458.5,54611.25,73298.75,50223,18252,7954.25,58212.75
8,7010,F,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,1,1,0,1,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,1,1,1,0,1,0,0,0,0,1,1,0,201.75,112.5,176.25,-18.25,-32.75,37.5,1126.25,2836.25,65.5,58.25,53927.75,33040.25,51370.75,1669,77.75,202.5,223942.75,182973.5,19613.5,16821
9,7023,M,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1047,768,833.5,3.25,81.25,85.5,6,192.75,234,34.25,193681.25,209817.5,247

In [12]:
columns = sorted(list(set(sinai.columns) & set(emory.columns)))
columns

['comorb_selfreport_1',
 'comorb_selfreport_10',
 'comorb_selfreport_11',
 'comorb_selfreport_12',
 'comorb_selfreport_13',
 'comorb_selfreport_14',
 'comorb_selfreport_15',
 'comorb_selfreport_16',
 'comorb_selfreport_17',
 'comorb_selfreport_18',
 'comorb_selfreport_2',
 'comorb_selfreport_3',
 'comorb_selfreport_4',
 'comorb_selfreport_5',
 'comorb_selfreport_6',
 'comorb_selfreport_7',
 'comorb_selfreport_8',
 'comorb_selfreport_9',
 'currentsymptoms_1',
 'currentsymptoms_10',
 'currentsymptoms_11',
 'currentsymptoms_12',
 'currentsymptoms_13',
 'currentsymptoms_15',
 'currentsymptoms_16',
 'currentsymptoms_17',
 'currentsymptoms_18',
 'currentsymptoms_19',
 'currentsymptoms_2',
 'currentsymptoms_20',
 'currentsymptoms_21',
 'currentsymptoms_22',
 'currentsymptoms_23',
 'currentsymptoms_24',
 'currentsymptoms_25',
 'currentsymptoms_26',
 'currentsymptoms_27',
 'currentsymptoms_28',
 'currentsymptoms_29',
 'currentsymptoms_3',
 'currentsymptoms_30',
 'currentsymptoms_31',
 'currents

In [16]:
columns = sorted(list(set(sinai.columns) & set(emory.columns)))
columns.append("yale_id")
merged = pd.concat([sinai[columns], emory[columns[:-1]]])
columns = list(filter(re.compile("currentsymptoms_.*").match, merged.columns))
columns = [int(val.split("_")[-1]) for val in columns]
columns = sorted(columns)
columns = [f"currentsymptoms_{i}" for i in columns]
columns_ = ['record_id', 'yale_id', 'sex']
columns_.extend(columns)
columns = list(filter(re.compile("comorb_selfreport_.*").match, merged.columns))
columns = [int(val.split("_")[-1]) for val in columns]
columns = sorted(columns)
columns = [f"comorb_selfreport_{i}" for i in columns]
columns_.extend(columns)
columns = columns_[:]
merged = merged[columns]
merged

,record_id,yale_id,sex,currentsymptoms_1,currentsymptoms_2,currentsymptoms_3,currentsymptoms_4,currentsymptoms_6,currentsymptoms_7,currentsymptoms_8,currentsymptoms_9,currentsymptoms_10,currentsymptoms_11,currentsymptoms_12,currentsymptoms_13,currentsymptoms_15,currentsymptoms_16,currentsymptoms_17,currentsymptoms_18,currentsymptoms_19,currentsymptoms_20,currentsymptoms_21,currentsymptoms_22,currentsymptoms_23,currentsymptoms_24,currentsymptoms_25,currentsymptoms_26,currentsymptoms_27,currentsymptoms_28,currentsymptoms_29,currentsymptoms_30,currentsymptoms_31,currentsymptoms_32,currentsymptoms_33,currentsymptoms_34,currentsymptoms_35,currentsymptoms_36,currentsymptoms_39,currentsymptoms_40,currentsymptoms_41,currentsymptoms_42,currentsymptoms_43,currentsymptoms_44,comorb_selfreport_1,comorb_selfreport_2,comorb_selfreport_3,comorb_selfreport_4,comorb_selfreport_5,comorb_selfreport_6,comorb_selfreport_7,comorb_selfreport_8,comorb_selfreport_9,comorb_selfreport_10,comorb_selfreport_11,comorb_selfreport_12,comorb_selfreport_13,comorb_selfreport_14,comorb_selfreport_15,comorb_selfreport_16,comorb_selfreport_17,comorb_selfreport_18
0,2,79,M,0,1,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3,NaN,F,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,1,0,0,0,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7,NaN,M,0,1,0,0,1,1,0,0,0,0,0,1,1,0,0,1,0,1,1,0,1,0,0,1,1,1,1,1,1,1,0,1,1,0,1,0,0,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8,NaN,F,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9,NaN,F,0,1,0,0,0,1,0,0,0,0,0,0,1,0,1,1,0,1,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,1,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55,3556,NaN,F,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
56,3582,NaN,F,0,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,1,1,0,1,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
57,3580,NaN,F,0,1,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0
58,3591,NaN,F,1,1,0,0,0,0,0,0,0,0,0,1,1,0,1,1,0,0,1,0,0,1,0,0,1,1,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [18]:
len(np.unique(merged["yale_id"].astype(str)))

135

In [19]:
# merged.to_csv("data_cleaned/emory_and_sinai_n885.csv", index = False)

In [ ]:
comorb = ["Hypertension", "Diabetes", "Chronic Kidney Disease", "Chronic Liver Disease (i.e Cirrhosis, Hepatitis B)", "Asthma", "Chronic Obstructive Pulmonary Disease", "Other Chronic Lung Disease", "Stroke", "Spinal Cord Injury", "Other Neurological Disease (Parkinson, Epilepsy, Dementia)", "Obesity", "Immunodeficiency (i.e auto-immune disease)", "Cancer", "Anxiety", "Depression", "Other Psychiatric Disease", "Eating disorder", "Inflammatory Bowel Disease/ Irritable Bowel Syndrome", "Other - please specify below", "None"]
comorb_dict = dict(zip(["comorb_selfreport_" + str(i) for i in range(1, len(comorb) + 1)], comorb))
del comorb_dict["comorb_selfreport_19"]
del comorb_dict["comorb_selfreport_20"]
comorb_dict

In [ ]:
clusters_15 = [8, 1, 1, 6, 8, 6, 8, 6, 6, 1, 8, 1, 8, 8, 1, 1, 6, 8, 1, 1, 1, 8, 1, 6, 1, 8, 1, 8, 8, 6, 6, 1, 6, 6, 1, 6, 8, 8, 6, 6, 6, 8, 6, 6, 1, 1, 1, 8, 1, 8, 8, 1, 6, 8, 6, 6, 1, 10, 6, 8, 1, 6, 8, 6, 1, 11, 1, 1, 1, 6, 1, 1, 6, 6, 6, 8, 8, 6, 6, 6, 6, 1, 6, 6, 1, 1, 1, 1, 6, 1, 10, 6, 7, 1, 1, 1, 7, 8, 1, 8, 6, 1, 8, 1, 1, 8, 4, 1, 1, 2, 1, 1, 9, 1, 2, 3, 2, 1, 4, 3, 10, 1, 1, 5, 4, 7, 3, 7, 11, 1, 1, 3, 2, 1, 2, 3, 2, 11, 1, 1, 7, 2, 2, 3, 2, 11, 1, 2, 3, 10, 2, 3, 1, 9, 4, 1, 5, 13, 3, 1, 3, 1, 1, 4, 1, 9, 14, 1, 1, 1, 10, 13, 9, 3, 10, 1, 1, 1, 1, 1, 1, 11, 1, 1, 1, 1, 10, 5, 3, 2, 1, 2, 1, 15, 1, 4, 2, 1, 11, 1, 3, 1, 1, 1, 15, 1, 14, 2, 5, 1, 1, 1, 3, 13, 12, 5, 1, 1, 7, 1, 5, 11, 5, 1, 2, 1, 1, 1, 1, 14, 1, 3, 1, 2, 5, 4, 1, 1, 4, 1, 5, 6, 1, 13, 1, 4, 2, 3, 1, 3, 1, 11, 4, 1, 7, 4, 1, 9, 1, 12, 5, 1, 1, 11, 3, 2, 1, 1, 2, 3, 1, 4, 4, 3, 5, 1, 4, 12, 13, 2, 10, 4, 1, 2, 3, 11, 1, 5, 7, 5, 11, 12, 4, 1, 3, 1, 1, 1, 1, 1, 9, 1, 13, 1, 2, 12, 2, 2, 1, 5, 7, 1, 5, 11, 5, 1, 4, 1, 3, 12, 13, 2, 4, 3, 1, 4, 1, 13, 4, 1, 1, 12, 3, 4, 1, 1, 1, 1, 1, 12, 1, 1, 2, 7, 9, 1, 9, 7, 3, 2, 1, 3, 2, 2, 4, 4, 7, 4, 1, 9, 15, 1, 2, 1, 10, 2, 7, 1, 1, 9, 1, 10, 3, 1, 1, 1, 1, 14, 2, 12, 3, 3, 1, 4, 15, 5, 5, 9, 5, 5, 2, 1, 14, 2, 1, 1, 13, 5, 1, 11, 1, 2, 1, 10, 13, 1, 2, 1, 1, 2, 3, 1, 1, 5, 9, 10, 10, 3, 9, 2, 13, 3, 1, 5, 3, 13, 1, 1, 2, 3, 1, 13, 9, 12, 3, 7, 2, 5, 14, 1, 2, 1, 10, 9, 5, 4, 15, 1, 10, 9, 1, 9, 4, 1, 1, 2, 2, 5, 1, 2, 1, 10, 1, 7, 7, 1, 7, 1, 14, 4, 1, 7, 10, 12, 1, 5, 1, 2, 4, 12, 5, 12, 9, 10, 2, 2, 5, 11, 13, 2, 7, 5, 5, 4, 14, 10, 1, 4, 15, 12, 10, 11, 3, 7, 2, 2, 1, 7, 7, 1, 10, 13, 4, 10, 1, 1, 14, 2, 7, 1, 11, 1, 4, 11, 12, 5, 1, 2, 1, 9, 1, 1, 1, 4, 12, 14, 1, 3, 14, 7, 1, 1, 1, 1, 2, 3, 12, 9, 7, 1, 12, 1, 9, 4, 2, 1, 4, 1, 2, 1, 5, 11, 9, 11, 4, 2, 12, 10, 5, 1, 5, 1, 1, 9, 3, 1, 2, 9, 2, 1, 1, 14, 4, 2, 9, 15, 1, 5, 1, 4, 2, 2, 4, 1, 2, 1, 2, 13, 3, 2, 12, 10, 13, 2, 2, 2, 2, 1, 1, 12, 5, 4, 11, 3, 13, 11, 3, 15, 7, 4, 3, 11, 2, 5, 1, 11, 3, 1, 3, 2, 4, 4, 2, 2, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 3, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 10, 1, 11, 1, 4, 5]
clusters_16 = [11, 1, 1, 10, 11, 13, 11, 13, 10, 1, 11, 1, 11, 11, 1, 1, 13, 11, 1, 1, 1, 11, 1, 10, 1, 11, 1, 11, 11, 13, 13, 10, 10, 10, 1, 13, 11, 11, 13, 13, 13, 11, 13, 13, 10, 1, 1, 11, 1, 11, 11, 1, 13, 11, 10, 10, 1, 8, 10, 11, 1, 13, 11, 10, 10, 16, 1, 1, 1, 13, 10, 1, 13, 13, 13, 10, 11, 13, 13, 13, 13, 10, 10, 10, 10, 1, 1, 1, 10, 10, 8, 13, 3, 1, 1, 1, 9, 11, 1, 11, 13, 1, 11, 1, 1, 11, 2, 1, 1, 3, 1, 1, 12, 1, 1, 3, 1, 1, 2, 1, 8, 1, 1, 6, 2, 3, 7, 3, 16, 5, 1, 7, 4, 1, 4, 1, 4, 16, 1, 1, 3, 4, 4, 7, 4, 4, 1, 1, 7, 8, 1, 2, 1, 12, 2, 1, 6, 14, 2, 1, 7, 1, 1, 12, 4, 15, 2, 1, 1, 5, 8, 14, 12, 2, 8, 1, 5, 1, 1, 1, 1, 16, 1, 1, 1, 1, 8, 6, 7, 3, 10, 5, 1, 7, 1, 2, 1, 1, 16, 1, 1, 1, 5, 1, 7, 1, 3, 4, 6, 1, 1, 1, 2, 14, 9, 6, 1, 1, 9, 1, 6, 4, 6, 1, 1, 1, 1, 1, 5, 3, 1, 7, 1, 3, 6, 2, 1, 1, 2, 1, 6, 10, 1, 14, 1, 2, 8, 1, 1, 1, 1, 4, 2, 5, 9, 2, 1, 15, 4, 9, 6, 1, 1, 16, 3, 3, 1, 1, 1, 2, 1, 12, 2, 2, 6, 1, 2, 9, 14, 1, 8, 2, 1, 3, 1, 4, 1, 6, 3, 6, 16, 9, 2, 1, 4, 1, 1, 1, 1, 1, 12, 1, 14, 4, 1, 9, 4, 1, 1, 6, 3, 1, 6, 4, 6, 1, 2, 1, 7, 14, 14, 1, 2, 2, 1, 2, 1, 14, 2, 5, 1, 14, 7, 2, 1, 1, 1, 1, 1, 9, 5, 1, 3, 3, 12, 4, 12, 3, 3, 5, 1, 1, 5, 1, 2, 2, 3, 2, 10, 12, 7, 5, 1, 1, 8, 3, 9, 1, 1, 12, 1, 8, 7, 1, 1, 1, 5, 15, 1, 15, 7, 7, 5, 2, 7, 6, 6, 12, 6, 6, 10, 1, 2, 1, 1, 5, 14, 6, 1, 4, 1, 4, 1, 8, 14, 1, 1, 1, 1, 5, 7, 1, 5, 6, 12, 8, 8, 3, 15, 1, 14, 1, 5, 6, 7, 14, 1, 1, 4, 1, 1, 14, 12, 9, 7, 3, 1, 6, 15, 5, 3, 1, 8, 12, 6, 12, 7, 1, 8, 15, 1, 12, 2, 5, 1, 3, 1, 6, 1, 4, 1, 8, 5, 3, 9, 5, 9, 4, 2, 2, 1, 3, 8, 15, 1, 5, 1, 3, 2, 9, 6, 14, 12, 8, 5, 5, 6, 16, 14, 1, 3, 6, 6, 2, 15, 8, 1, 2, 7, 9, 8, 16, 7, 3, 5, 3, 1, 3, 3, 1, 8, 14, 2, 8, 1, 5, 8, 3, 9, 1, 4, 1, 2, 4, 9, 4, 1, 4, 1, 12, 1, 5, 1, 2, 9, 3, 5, 7, 9, 9, 1, 5, 1, 1, 1, 7, 9, 12, 3, 1, 9, 4, 15, 2, 5, 1, 7, 5, 3, 1, 6, 4, 12, 4, 2, 5, 9, 8, 6, 1, 6, 1, 1, 12, 7, 4, 4, 12, 5, 1, 1, 15, 2, 3, 15, 7, 1, 6, 1, 2, 4, 5, 12, 5, 3, 1, 4, 14, 7, 10, 14, 8, 14, 1, 1, 5, 4, 1, 1, 9, 6, 2, 4, 7, 14, 4, 3, 7, 9, 2, 7, 16, 3, 6, 1, 16, 4, 1, 7, 3, 2, 12, 1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 1, 1, 1, 2, 7, 1, 1, 1, 1, 1, 1, 1, 1, 7, 1, 1, 1, 1, 1, 1, 8, 8, 1, 16, 1, 2, 3]

In [ ]:
clusters = clusters_16[:]
surveys = pd.read_csv("emory_and_sinai_n675.csv")#.astype(str)
surveys["cluster"] = clusters
print(surveys["sex"].value_counts())
surveys

In [ ]:
covariate = "sex"
keys = list(np.unique(surveys[covariate].astype(str)))
temp = surveys[[covariate]]
temp["cluster"] = clusters
df = pd.DataFrame()
for cluster in np.unique(temp["cluster"]): 
    subset = temp[temp["cluster"] == cluster]
    dictionary = dict(subset.value_counts(covariate))
    key1 = keys[0]
    key2 = keys[1]
    key3 = "nan"
    dict_df = {"cluster": cluster, key1: 0, key2: 0, key3:0}
    if len(dictionary) == 2: 
        dict_df[key1] = dictionary[key1]
        dict_df[key2] = dictionary[key2]
    elif key1 in dictionary: 
        dict_df[key1] = dictionary[key1]
    elif key2 in dictionary: 
        dict_df[key2] = dictionary[key2]
    else: 
        dict_df[key3] = dict(subset.isna()[covariate].value_counts())[True]
    if sum(dictionary.values()) != subset.shape[0]: 
        dict_df[key3] = dict(subset.isna()[covariate].value_counts())[True]
    df = pd.concat([df, pd.DataFrame(dict_df, index = [0])])
# df = df.apply(lambda x: round(x, 2))
if covariate == "sex": 
    df = df[["cluster", "F", "M", "nan"]]
    values_m = df["M"] / (df["M"] + df["F"] + df["nan"])
    values_f = df["F"] / (df["M"] + df["F"] + df["nan"])
    values_nan = df["nan"] / (df["M"] + df["F"] + df["nan"])
    df["M"] = values_m
    df["F"] = values_f
    df["nan"] = values_nan
else: 
    df = df[["cluster", 1, 0, "nan"]]
if df["nan"].sum() == 0: del df["nan"]
df

In [ ]:
columns = ["C" + str(val) for val in df["cluster"]]
dictionary = dict(df.iloc[:,1:])

fig, ax = plt.subplots(layout='constrained', figsize = (8, 6))
x = np.arange(len(columns)) 
width = 0.6
multiplier = 0
bottom = np.zeros(df.shape[0])
colors = [sns.color_palette("Reds")[2:3], sns.color_palette("Blues")[2:3], "#d3d3d3"]

for attribute, measurement in dictionary.items(): 
    measurement = [round(val, 3) for val in measurement]
    if covariate == "bmi_cat_binary": 
        if attribute == 0: attribute = '<25, 25_to_30'
        elif attribute == 1: attribute = ">_30"
    elif covariate == "calcage_binary": 
        if attribute == 0: attribute = "<=50"
        elif attribute == 1: attribute = ">50"
    elif covariate == "income_binary": 
        if attribute == 0: attribute = "<=$100k"
        elif attribute == 1: attribute = ">$100k"
    elif "_binary" not in covariate: 
        if attribute == 0: attribute = "no " + covariate
        elif attribute == 1: attribute = covariate
    print(attribute, list(measurement))
    rects = ax.bar(x, measurement, width, label=attribute, bottom = bottom, color = colors[int(multiplier%3)])
    ax.bar_label(rects, label_type = "center", rotation = 0)
    multiplier += 1
    bottom += measurement

if covariate == "sex": 
    ax.plot([-0.5, 15.5], [0.5, 0.5], color = "grey")
    ax.set_ylabel('Percentage')
    ax.set_title('Percentage of ' + covariate + ' in each cluster')
else: 
    if covariate not in ["bmi_cat_binary", "income_binary", "calcage_binary", "smk", "student"]: 
        ax.set_ylim(0, 65)
    ax.set_ylabel('Number of patients')
    ax.set_title('Number of ' + covariate + ' in each cluster')
ax.set_xlabel("Cluster")
ax.legend(loc='upper left', ncols=1)
ax.set_xticks(x, columns, rotation = 0)
plt.show()

In [ ]:
conditions = merged.copy()
conditions["cluster"] = ["C" + str(c) for c in clusters_16]
dictionary = dict(conditions["cluster"].value_counts())
df = pd.DataFrame({"cluster": dictionary.keys(), "total": dictionary.values()})
df = df.sort_values(["total", "cluster"], ascending = [False, True])
for condition in comorb_dict.keys(): 
    patients = list(conditions[conditions[condition] == 1.0].index)
    dictionary = dict(conditions.iloc[patients, :]["cluster"].value_counts())
    df2 = pd.DataFrame({"cluster": list(dictionary.keys()), comorb_dict[condition]: list(dictionary.values())})
    df = df.merge(df2, on = "cluster", how = "outer")
    df["proportion_" + comorb_dict[condition]] = df[comorb_dict[condition]] / df["total"]
# df.to_csv("comorb_proportions_emory_sinai_15clusters.csv", index = False)
# df.to_csv("comorb_proportions_emory_sinai_16clusters.csv", index = False)
df

In [ ]:
conditions = merged.copy()
conditions["cluster"] = ["C" + str(c) for c in clusters]
dictionary = dict(conditions["cluster"].value_counts())
df = pd.DataFrame({"cluster": dictionary.keys(), "total": dictionary.values()})
df = df.sort_values(["total", "cluster"], ascending = [False, True])
for condition in comorb_dict.keys(): 
    patients = list(conditions[conditions[condition] == 1.0].index)
    dictionary = dict(conditions.iloc[patients, :]["cluster"].value_counts())
    df2 = pd.DataFrame({"cluster": list(dictionary.keys()), comorb_dict[condition]: list(dictionary.values())})
    df = df.merge(df2, on = "cluster", how = "outer")
    df["proportion_" + comorb_dict[condition]] = df[comorb_dict[condition]] / df["total"]
# df.to_csv("comorb_proportions_sinai.csv", index = False)
df

In [ ]:
# merged.to_csv("data_cleaned/emory_and_sinai_n885.csv", index = False)

# serum and mensa analysis

In [ ]:
print(data.shape)
cluster_dict = {3633: 4, 7191: 3, 7283: 7, 7385: 9, 7415: 2, 7423: 7, 7424: 16, 7426: 3, 7010: 6, 7023: 1, 7024: 16, 7030: 4, 7033: 1, 7034: 7, 7032: 3, 7053: 2, 7069: 12, 7075: 1, 7076: 1, 7077: 7, 7090: 1, 3198: 1, 3199: 1, 3231: 1, 3232: 1, 3234: 1, 3260: 1, 3264: 1, 3290: 5, 3292: 1, 3299: 1, 3320: 1, 3322: 1, 3323: 1, 3324: 1, 3339: 1, 3341: 2, 3346: 7, 3362: 1, 3365: 1, 3381: 1, 3386: 1, 3391: 1, 3400: 1, 3401: 1, 3404: 1, 3427: 7, 3438: 1, 3459: 1, 3460: 1, 3471: 1, 3474: 1, 3475: 1, 3503: 8, 3532: 8, 3556: 1, 3582: 16, 3580: 1, 3591: 2, 3593: 3}
data["cluster"] = [cluster_dict[int(val)] for val in data["record_id"]]
data

In [ ]:
cluster_severity = {"mild": [1], "moderate": [5, 4, 10, 11, 6, 8, 3, 13, 16], "severe": [2, 7, 12, 9, 15, 14]}
for key in list(cluster_severity.keys()): 
    for val in cluster_severity[key]: cluster_severity[val] = key
    del cluster_severity[key]
data["severity"] = [cluster_severity[int(val)] for val in data["cluster"]]
data["severity"].value_counts()

In [ ]:
import math
columns = ["record_id", "cluster", "severity"]
columns.extend(igg_cols)
df_igg = data[columns].copy()
df_igg = df_igg.set_index(["record_id", "cluster", "severity"])
df_igg = df_igg.astype(float)
df_igg = df_igg - df_igg.min() + 1
for col in df_igg.columns: 
    df_igg[col] = [math.log2(v) for v in df_igg[col]]
df_igg = df_igg.reset_index()
df_igg

In [ ]:
# Calculating mensa/serum mean values per cluster
df = pd.DataFrame()
df["sum"] = [2.693430126565952, 15.267783388134077, 13.622444758818038, 9.349821437001479, 7.531502417733883, 11.455694586117241, 16.085832351121763, 13.575811994776776, 23.02057402795433, 10.143111441440476, 11.0002448479398, 21.085280348374372, 14.941260304234193, 31.910781562092247, 28.21574051988618, 15.158387731989784]
df.index = ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16']
df["cluster"] = [int(val.replace("C", "")) for val in df.index]

for col in df_igg.columns: 
    if "mensa" not in col and "serum" not in col: 
        continue
    values = {}
    for cluster in np.unique(df_igg["cluster"]): 
        subset = df_igg[df_igg["cluster"] == cluster]
        values[cluster] = subset[col].mean()
#     print(values)
    df["mean_" + col] = [values[c] if c in values else float('nan') for c in df['cluster']]
# df.to_csv("severity_sum_mean_mensa_serum.csv", index = False)
df

In [ ]:
import statannot
import seaborn as sns
import scipy.stats as stats
columns = list(filter(re.compile(".*_mensa_.*").match, df_igg.columns))
# columns = list(filter(re.compile(".*_serum_.*").match, df_igg.columns))
df = df_igg.melt(id_vars = ["record_id", "cluster", "severity"])
df = df[df["variable"].isin(columns)]
# df = df.replace("moderate", "severe")
df = df.rename(columns = {"variable": " ", "value": "log2-transformed values"})
df

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(10, 5))
ax = sns.boxplot(df, x = " ", y = "log2-transformed values", hue = "severity", 
                 palette = {"mild": "green", "moderate": "yellow", "severe": "red"}, 
#                  hue_order = ["mild", "severe"], 
                 hue_order = ["mild", "moderate", "severe"], 
                )
a = plt.xticks(rotation = 80)
sns.move_legend(ax, "upper left", bbox_to_anchor = (1, 1))
box_pairs = []
p_values = []
pvalue_thresholds = []
# for var in np.unique(df[" "]): 
#     box_pairs.append(((var, "mild"), (var, "moderate")))
#     box_pairs.append(((var, "moderate"), (var, "severe")))
#     box_pairs.append(((var, "severe"), (var, "mild")))
for var in np.unique(df[" "]): 
    box_pairs.append(((var, "mild"), (var, "severe")))
    subset = df[df[" "] == var]
    s_1 = subset[subset["severity"] == "mild"]["log2-transformed values"]
    s_2 = subset[subset["severity"] == "moderate"]["log2-transformed values"]
    s_3 = subset[subset["severity"] == "severe"]["log2-transformed values"]
    f_statistic, p_value = stats.f_oneway(s_1, s_2, s_3)
    p_values.append(p_value)
    print(var, p_value)
    print(f_statistic)
#     pvalue_thresholds.append([p_value, round(p_value, 6)])
#     print(p_value)
# pvalue_thresholds.append([6.500e-03, "**"])
pvalue_thresholds = [[1e-4, "****"], [1e-3, "***"], [1e-2, "**"], [0.05, "*"], [1, "ns"]]
# pvalue_thresholds = [[0.55, "*"], [1, "ns"]]
# pvalue_thresholds = [[0.27, "*"], [1, "ns"]]
test_results = statannot.add_stat_annotation(ax, data = df, x = " ", y = "log2-transformed values", 
                                             perform_stat_test = False, pvalues = p_values, #test_short_name = "ANOVA test", 
                                             hue = "severity", 
#                                              hue_order = ["mild", "severe"], 
                                             hue_order = ["mild", "moderate", "severe"], 
                                             box_pairs = box_pairs, #test = 't-test_ind', 
                                             pvalue_thresholds = pvalue_thresholds, 
                                             text_format = 'star', loc = 'outside', verbose = 2)

Yes we need to do the correction. But for ANOVA, maybe we can use Tukey HSD.  Bonferroni may kill all significance - I don't know...

Also, double check the ANOVA and make sure the outputs are correct, which should include F values and degrees of freedom (within-group and between-group) before outputting p values.
 
make sure you use OLS when running ANOVA because our severity groups are ordered.
 
Based on the box plots, reviewers may ask us to do Kruskal-Wallis test instead of ANOVA because the data is a little skewed (especially the serum values) with outliers. ANOVA assume a Gaussian distribution.
